# Hearthstone Card Generator – Main Training Notebook

This notebook walks through the full training pipeline:
1. Configuration
2. Data loading & exploration
3. Model construction (CLIP encoder + GNN)
4. Training loop
5. Evaluation & visualisation

In [ ]:
import sys
import os

# Add repo root to path when running from the experiments/ directory
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))

import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

from models.clip_encoder import CLIPEncoder
from models.gnn_model import GNNModel
from utils.data_loader import HearthstoneDataLoader
from utils.preprocessing import preprocess_card_data

print('PyTorch version:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Configuration

In [ ]:
CONFIG = {
    'data_dir': '../data',          # Path to dataset root
    'clip_model': 'openai/clip-vit-base-patch32',
    'projection_dim': 256,
    'hidden_channels': 256,
    'out_channels': 128,
    'batch_size': 32,
    'num_workers': 2,
    'lr': 1e-4,
    'weight_decay': 1e-5,
    'epochs': 50,
    'seed': 42,
    'output_dir': '../checkpoints',
}

torch.manual_seed(CONFIG['seed'])
print('Config loaded.')

## 2. Data Loading

In [ ]:
loader_factory = HearthstoneDataLoader(
    data_dir=CONFIG['data_dir'],
    clip_model_name=CONFIG['clip_model'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)

loaders = loader_factory.get_all_loaders()
print(f"Train batches: {len(loaders['train'])}")
print(f"Val   batches: {len(loaders['val'])}")
print(f"Test  batches: {len(loaders['test'])}")

## 3. Model Construction

In [ ]:
clip_encoder = CLIPEncoder(
    model_name=CONFIG['clip_model'],
    projection_dim=CONFIG['projection_dim'],
    freeze_clip=True,
).to(device)

# GNN input dim = projection_dim + stat_features (3) + one_hot_features (18)
gnn_in = CONFIG['projection_dim'] + 3 + 18

gnn_model = GNNModel(
    in_channels=gnn_in,
    hidden_channels=CONFIG['hidden_channels'],
    out_channels=CONFIG['out_channels'],
).to(device)

total_params = sum(p.numel() for p in gnn_model.parameters() if p.requires_grad)
print(f'GNN trainable parameters: {total_params:,}')

## 4. Training Loop

In [ ]:
import torch.nn as nn
import torch.optim as optim
from pathlib import Path

criterion = nn.CrossEntropyLoss()

params = list(filter(lambda p: p.requires_grad, clip_encoder.parameters()))
params += list(gnn_model.parameters())
optimizer = optim.AdamW(params, lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])

train_losses, val_losses = [], []
best_val_loss = float('inf')
output_dir = Path(CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

for epoch in range(1, CONFIG['epochs'] + 1):
    # --- Training ---
    clip_encoder.train()
    gnn_model.train()
    train_loss = 0.0
    for batch in tqdm(loaders['train'], desc=f'Epoch {epoch} [train]', leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch.get('label')
        pixel_values = batch.get('pixel_values')

        if pixel_values is not None:
            pixel_values = pixel_values.to(device)
            img_emb, txt_emb = clip_encoder(pixel_values, input_ids, attention_mask)
            feats = (img_emb + txt_emb) / 2
        else:
            _, feats = clip_encoder(None, input_ids, attention_mask)

        if labels is not None:
            labels = labels.to(device)
            logits = gnn_model.fc(feats)
            loss = criterion(logits, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

    train_loss /= max(len(loaders['train']), 1)
    train_losses.append(train_loss)

    # --- Validation ---
    clip_encoder.eval()
    gnn_model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch in loaders['val']:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch.get('label')
            pixel_values = batch.get('pixel_values')

            if pixel_values is not None:
                pixel_values = pixel_values.to(device)
                img_emb, txt_emb = clip_encoder(pixel_values, input_ids, attention_mask)
                feats = (img_emb + txt_emb) / 2
            else:
                _, feats = clip_encoder(None, input_ids, attention_mask)

            if labels is not None:
                labels = labels.to(device)
                logits = gnn_model.fc(feats)
                loss = criterion(logits, labels)
                val_loss += loss.item()

    val_loss /= max(len(loaders['val']), 1)
    val_losses.append(val_loss)
    scheduler.step()

    print(f'Epoch {epoch:03d} | train={train_loss:.4f} | val={val_loss:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'clip_encoder': clip_encoder.state_dict(),
            'gnn_model': gnn_model.state_dict(),
            'val_loss': val_loss,
        }, output_dir / 'best_model.pt')
        print(f'  ✓ New best checkpoint saved.')

## 5. Training Curve Visualisation

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Curves')
plt.legend()
plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150)
plt.show()